# BƯỚC 2 — EDA và chất lượng dữ liệu
**Đề tài 5 — Bệnh viện: Dự đoán tái nhập viện trong 30 ngày**

**Câu hỏi nghiệp vụ:** Sau khi xuất viện, nên gọi theo dõi cho bệnh nhân nào trước?

Notebook này **chỉ khảo sát** dữ liệu, không xóa bản ghi hay áp dụng phương pháp làm sạch. Đầu vào `data/processed/step1_data.csv` từ Bước 1. Kết quả xuất vào `reports/eda/`.

Chạy các ô theo thứ tự bằng **Run → Run All Cells**.


## 1. Nạp dữ liệu và tạo thư mục kết quả

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Có thể khởi động JupyterLab ở thư mục gốc hoặc mở notebook từ thư mục notebooks.
cwd = Path.cwd()
project_root = next((p for p in [cwd, *cwd.parents] if (p / 'data' / 'processed' / 'step1_data.csv').exists()), None)
if project_root is None:
    raise FileNotFoundError('Không tìm thấy data/processed/step1_data.csv. Mở JupyterLab ở D:/Readmission_Project và kiểm tra file Bước 1.')
source = project_root / 'data' / 'processed' / 'step1_data.csv'
out = project_root / 'reports' / 'eda'
out.mkdir(parents=True, exist_ok=True)

# pandas nhận các ô trống thành NaN; dấu ? được giữ lại để đếm riêng.
df = pd.read_csv(source, keep_default_na=True)
assert {'encounter_id', 'patient_nbr', 'readmitted', 'readmitted_30d'}.issubset(df.columns)
print('File:', source)
print('Kích thước:', df.shape)
print('Thư mục kết quả:', out)
display(df.head())

## 2. Tổng quan dữ liệu và thống kê mô tả

In [ ]:
print('Số bệnh nhân duy nhất:', df['patient_nbr'].nunique())
print('Số lượt khám:', len(df))
print('Số encounter_id trùng:', df['encounter_id'].duplicated().sum())
print('Số dòng trùng hoàn toàn:', df.duplicated().sum())
display(df.describe(include='all').T)

numeric_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                'num_medications', 'number_outpatient', 'number_emergency',
                'number_inpatient', 'number_diagnoses']
summary = df[numeric_cols].describe().T
summary.to_csv(out / 'numeric_summary.csv', encoding='utf-8-sig')
display(summary)

## 3. Đánh giá giá trị thiếu — `NaN`, chuỗi rỗng và dấu `?`
Chú ý: `max_glu_serum` và `A1Cresult` có thể không được đo; cần phân biệt **không xét nghiệm** với lỗi thu thập, không tự thay bằng kết quả bình thường.

In [ ]:
# Đếm dữ liệu thiếu theo định dạng thực tế, KHÔNG chỉnh sửa df.
missing_rows = []
for c in df.columns:
    s = df[c]
    na_count = int(s.isna().sum())
    q_count = int(s.astype('string').str.strip().eq('?').fillna(False).sum())
    blank_count = int(s.astype('string').str.strip().eq('').fillna(False).sum())
    total = na_count + q_count + blank_count
    missing_rows.append([c, na_count, q_count, blank_count, total, round(100 * total / len(df), 2)])
missing = pd.DataFrame(missing_rows, columns=['column', 'na_count', 'question_count', 'blank_count', 'total_missing', 'missing_pct'])
missing = missing.sort_values('total_missing', ascending=False)
display(missing.head(20))
missing.to_csv(out / 'missing_report.csv', index=False, encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(10, 6))
top = missing.query('total_missing > 0').head(12).sort_values('missing_pct')
ax.barh(top['column'], top['missing_pct'])
ax.set(title='Top thuộc tính có dấu hiệu thiếu dữ liệu', xlabel='Tỷ lệ (%)')
fig.tight_layout()
fig.savefig(out / '01_missing.png', dpi=140)
plt.show()

## 4. Kiểm tra nhãn và mất cân bằng lớp
`readmitted_30d = 1` khi giá trị gốc `readmitted` là `<30`; ngược lại bằng 0. Kiểm tra lại mapping từ đầu.

In [ ]:
print('Phân bố nhãn gốc:')
display(df['readmitted'].value_counts(dropna=False))
expected_target = df['readmitted'].eq('<30').astype(int)
mismatch = int((df['readmitted_30d'] != expected_target).sum())
print('Số dòng sai mapping nhãn:', mismatch)
if mismatch:
    print('CẢNH BÁO: Nhãn tạo ở Bước 1 chưa khớp với quy tắc <30.')
target_counts = df['readmitted_30d'].value_counts().sort_index()
print('Tỷ lệ nhãn 1: {:.2f}%'.format(df['readmitted_30d'].mean() * 100))
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Không <30 ngày (0)', '<30 ngày (1)'], [target_counts.get(0,0), target_counts.get(1,0)])
ax.set(title='Phân bố nhãn tái nhập viện', ylabel='Số lượt nhập viện')
fig.tight_layout(); fig.savefig(out / '02_target.png', dpi=140); plt.show()

## 5. Phân bố độ tuổi, giới tính và thời gian nằm viện

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
age_counts = df['age'].value_counts().sort_index()
axes[0].bar(age_counts.index.astype(str), age_counts.values)
axes[0].tick_params(axis='x', rotation=65)
axes[0].set_title('Số lượt nhập viện theo nhóm tuổi')
gender_counts = df['gender'].value_counts(dropna=False)
axes[1].bar(gender_counts.index.astype(str), gender_counts.values)
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_title('Giới tính (bao gồm mã lạ nếu có)')
axes[2].hist(df['time_in_hospital'].dropna(), bins=np.arange(0.5, 15.6, 1))
axes[2].set(title='Số ngày nằm viện', xlabel='Ngày', ylabel='Lượt')
fig.tight_layout(); fig.savefig(out / '03_demographics_stay.png', dpi=140); plt.show()

## 6. Phát hiện giá trị bất thường, nhóm hiếm và giá trị không hợp lệ
`IQR` là tín hiệu để xem xét, **không đồng nghĩa với dữ liệu sai**. Các giá trị cao có thể là bệnh nhân cần ưu tiên và không được tự động xoá.

In [ ]:
quality = []
for c in numeric_cols:
    s = pd.to_numeric(df[c], errors='coerce')
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3-q1
    low, high = q1-1.5*iqr, q3+1.5*iqr
    quality.append([c, int((s < 0).sum()), int(((s < low) | (s > high)).sum()), float(s.min()), float(s.max())])
quality = pd.DataFrame(quality, columns=['column', 'negative_count', 'iqr_flag_count', 'min', 'max'])
display(quality)
print('Phân bố mã giới tính:')
display(df['gender'].value_counts(dropna=False))
print('Cột có 1 giá trị duy nhất (sau khi giữ nguyên dữ liệu):')
display(df.nunique(dropna=False).loc[lambda s: s <= 1])
quality.to_csv(out / 'numeric_quality_flags.csv', index=False, encoding='utf-8-sig')

## 7. Tỷ lệ tái nhập viện theo các nhóm
Các kết quả mang tính **mô tả/tương quan**, không chứng minh quan hệ nhân quả. Cần xem cả `n` ở mỗi nhóm.

In [ ]:
def group_rate(col):
    return (df.groupby(col, dropna=False)['readmitted_30d']
            .agg(n='size', readmit_count='sum', rate='mean')
            .assign(rate_pct=lambda x: (x['rate']*100).round(2))
            .sort_values('n', ascending=False))
for col in ['age', 'gender', 'race', 'time_in_hospital']:
    result = group_rate(col)
    print('\n', col)
    display(result.head(20))
    result.to_csv(out / f'readmission_by_{col}.csv', encoding='utf-8-sig')

age_rate = group_rate('age').sort_index()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(age_rate.index.astype(str), age_rate['rate_pct'])
ax.set(title='Tỷ lệ tái nhập viện <30 ngày theo nhóm tuổi', ylabel='Tỷ lệ (%)', xlabel='Nhóm tuổi')
ax.tick_params(axis='x', rotation=50)
fig.tight_layout(); fig.savefig(out / '04_age_readmission.png', dpi=140); plt.show()

## 8. Nhật ký quyết định và kiểm tra tối thiểu 3 vấn đề chất lượng dữ liệu

In [ ]:
# Ghi nhận từ dữ liệu thực tế, KHÔNG bịa số liệu hay kết luận xóa cột ngay.
checks = pd.DataFrame([
    ['Dấu ? trong weight', int(df['weight'].astype('string').str.strip().eq('?').fillna(False).sum()), 'Ghi là missing; quyết định giữ/bỏ cột ở Bước 3 sau khi cân nhắc'],
    ['Thiếu medical_specialty', int(missing.set_index('column').loc['medical_specialty','total_missing']), 'Không suy diễn specialty khi chưa có căn cứ'],
    ['Thiếu payer_code', int(missing.set_index('column').loc['payer_code','total_missing']), 'Xem khả năng xử lý unknown như nhóm riêng'],
    ['Không có kết quả A1Cresult', int(missing.set_index('column').loc['A1Cresult','total_missing']), 'Cần kiểm tra ý nghĩa không xét nghiệm'],
    ['Không có kết quả max_glu_serum', int(missing.set_index('column').loc['max_glu_serum','total_missing']), 'Không coi mặc định là mức đường huyết bình thường'],
    ['Lặp patient_nbr (số lượt vượt quá số bệnh nhân)', int(len(df)-df['patient_nbr'].nunique()), 'Ở bước phân lớp phải chia tập theo bệnh nhân để tránh leakage'],
    ['Sai mapping nhãn readmitted_30d', mismatch, 'Nếu >0 cần sửa logic nhãn'],
], columns=['van_de', 'so_luong', 'huong_xu_ly_de_xuat'])
display(checks)
checks.to_csv(out / 'data_quality_findings.csv', index=False, encoding='utf-8-sig')
decisions = pd.DataFrame([
    ['B2-01', 'Không tự động loại các dòng có ?', 'Chưa biết cơ chế thiếu; tránh thiên lệch mẫu', 'Số dòng/ước lượng tỷ lệ tái nhập', 'Chờ Bước 3'],
    ['B2-02', 'Giữ nguyên giá trị rất cao để đánh giá', 'Outlier thống kê không luôn là lỗi nhập liệu', 'Phân phối đặc trưng và mô hình', 'Chờ Bước 3'],
    ['B2-03', 'Đánh giá sau này theo patient_nbr', 'Một bệnh nhân có thể có nhiều lượt', 'Chỉ số validation/test', 'Đã quyết định'],
    ['B2-04', 'Không dùng accuracy đơn độc', 'Lớp dương chiếm tỷ lệ nhỏ', 'Đánh giá baseline, recall, precision, PR-AUC', 'Đã quyết định'],
], columns=['id', 'quyet_dinh', 'ly_do', 'anh_huong', 'trang_thai'])
display(decisions)
decisions.to_csv(out / 'decision_log_step2.csv', index=False, encoding='utf-8-sig')
print('Đã xuất các CSV và PNG vào:', out)
print('Bước tiếp theo: Bước 3 — làm sạch dữ liệu (không ghi đè file gốc).')

## Kết luận cần đưa vào báo cáo
- Bảng phân tích missing và **ít nhất 3 vấn đề có số đếm thực tế** nằm trong `data_quality_findings.csv`.
- Các biểu đồ PNG có thể đưa vào báo cáo/slide; các bảng phân nhóm có `n` và `rate_pct`.
- Ghi rõ rằng `?` không được `df.isna()` mặc định nhận ra; các trường không có xét nghiệm cần giải thích theo định nghĩa dữ liệu.
- **Không** xem IQR outlier là bản ghi sai hoặc tự kết luận tuổi/giới tính gây tái nhập viện.
- **Không** dùng bản khảo sát này để fit preprocessing hay mô hình; ở bước sau sẽ tách train/test theo `patient_nbr`.
